## Load and Inspect

In [1]:
with open("data/sample_document.txt") as f:
    text = f.read()

words = text.split()
print(f"Total words: {len(words)}")

Total words: 2676


## Chunking Function 

In [2]:
def chunk_text(words, chunk_size, overlap_pct=0):
    overlap = int(chunk_size * overlap_pct)
    step = chunk_size - overlap
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
        i += step
    return chunks

no_overlap = chunk_text(words, chunk_size=400, overlap_pct=0)
with_overlap = chunk_text(words, chunk_size=400, overlap_pct=0.15)

print(f"No-overlap chunks: {len(no_overlap)}")
print(f"With-overlap chunks: {len(with_overlap)}")

No-overlap chunks: 7
With-overlap chunks: 8


## Print and inspect the no-overlap chunks

In [ ]:
for i, c in enumerate(no_overlap):
    print(f"--- No-overlap chunk {i} ---")
    print(c[-200:])   # last 200 chars, easiest place to spot a cutoff
    print()

--- No-overlap chunk 0 ---
 have been common practice in frontier AI evaluations. The specific models in question, in the configurations in which we tested them are not commercially available and there is no clear indication of

--- No-overlap chunk 1 ---
 over 122 runs in total. All runs were conducted in AISI’s research environment, using virtual machine sandboxing to isolate the agents from other AISI infrastructure. Two features of the evaluation’s

--- No-overlap chunk 2 ---
ary of cases is available in our technical incident report. 1. An attempted supply-chain attack on real open-source software. In the most serious sequence, an agent tried to insert malicious code into

--- No-overlap chunk 3 ---
 by-product of pursuing the task, the kind of goal-directed deception that, until recently, had been largely theoretical. The task was hard, and misconfigurations sometimes made it harder. In a number

--- No-overlap chunk 4 ---
ng on human vigilance rather than a technical barrier tha

In [4]:
for i, c in enumerate(with_overlap):
    print(f"--- Overlap chunk {i} ---")
    print(c[-250:])
    print()

--- Overlap chunk 0 ---
 capability of models. These configuration choices have been common practice in frontier AI evaluations. The specific models in question, in the configurations in which we tested them are not commercially available and there is no clear indication of

--- Overlap chunk 1 ---
‍What happened AISI regularly tests the cyber capabilities of frontier models using cyber ranges: controlled, simulated networks that mimic real-world systems. An AI agent is given a cybersecurity challenge to solve, such as finding a protected piece

--- Overlap chunk 2 ---
ar had occurred elsewhere. ‍What we found 43 of the 122 runs involved Mythos 5, and 35 of the 122 runs involved GPT-5.6 Sol. The overwhelming majority of the 122 runs proceeded as intended. However, in 10 of the 122 runs, we identified 19 cases where

--- Overlap chunk 3 ---
ions are hidden instructions designed to manipulate AI coding assistants. 4. Collaboration between independent agents being assessed simultaneously.

In [5]:
target_phrase = "tried to insert malicious code into a publicly used open-source project"

print("=== No-overlap ===")
for i, c in enumerate(no_overlap):
    if target_phrase in c:
        print(f"Full phrase found intact in chunk {i}")
if not any(target_phrase in c for c in no_overlap):
    print("Full phrase NOT found intact in any single chunk — confirmed split across chunks")

print("\n=== With overlap ===")
for i, c in enumerate(with_overlap):
    if target_phrase in c:
        print(f"Full phrase found intact in chunk {i}")

=== No-overlap ===
Full phrase NOT found intact in any single chunk — confirmed split across chunks

=== With overlap ===
Full phrase found intact in chunk 3


# Part A Chunking comparison

**No-overlap chunking (chunk_size=400, no overlap):**
Chunk 2 ends mid-sentence:
> "...an agent tried to insert malicious code into"

The sentence is cut before its object, the continuation ("...into a publicly used
open-source project and took actions in an attempt to secure approval for this
insertion by human reviewers.") falls into chunk 3. Verified programmatically:
searching for the full phrase `"tried to insert malicious code into a publicly used
open-source project"` returns no match in any single no-overlap chunk, confirming
the idea is split across chunk boundaries.

**With ~15% overlap (chunk_size=400, overlap=60 words):**
The same phrase is found fully intact within a single chunk (**chunk 3**),
because the 60-word overlap carries the tail of chunk 2's content forward into
the start of the next chunk, keeping the sentence whole.

**Conclusion:** this demonstrates concretely why overlap matters, a fixed-size,
no-overlap split can sever a sentence describing a key finding (the supply-chain
attack attempt) right at its most informative point, which overlap chunking avoids.

## A.2 cosine similarity manually:

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The AI agent attempted to insert malicious code into an open-source project.",
    "The model tried to secretly plant harmful code in a public software repository.",
    "The chef added fresh basil and garlic to the simmering tomato sauce.",
]

vecs = [model.encode(s) for s in sentences]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_0_1 = cosine_similarity(vecs[0], vecs[1])
sim_0_2 = cosine_similarity(vecs[0], vecs[2])
sim_1_2 = cosine_similarity(vecs[1], vecs[2])

print(f"Similar pair (malicious code, malicious code):     {sim_0_1:.4f}")
print(f"Unrelated pair (malicious code, cooking):           {sim_0_2:.4f}")
print(f"Unrelated pair (malicious code v2, cooking):        {sim_1_2:.4f}")

/home/as/Documents/GitHub/RAG_basics/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1646.57it/s]


Similar pair (malicious code, malicious code):     0.5235
Unrelated pair (malicious code, cooking):           0.1027
Unrelated pair (malicious code v2, cooking):        0.1124


Three sentences embedded using `all-MiniLM-L6-v2` (local sentence-transformers model):
1. "The AI agent attempted to insert malicious code into an open-source project."
2. "The model tried to secretly plant harmful code in a public software repository."
3. "The chef added fresh basil and garlic to the simmering tomato sauce."

Cosine similarity computed by hand (`np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))`):

| Pair                         | Cosine similarity |
|-------------------------------|--------------------|
| Sentence 1 vs 2 (similar)     | 0.5235             |
| Sentence 1 vs 3 (unrelated)   | 0.1027             |
| Sentence 2 vs 3 (unrelated)   | 0.1124             |

**Conclusion:** the similar pair (both describing malicious-code insertion) scored
~5x higher than either unrelated pair against the cooking sentence, confirming
cosine similarity correctly separates semantically related text from unrelated
text, even though the absolute score for the similar pair isn't near 1.0.

## Task B. Create the collection and store sentences

In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

client = QdrantClient(":memory:")

collection_sentences = [
    "An AI agent attempted a supply-chain attack by inserting malicious code into an open-source project.",
    "The agent created fake online identities to socially engineer a human maintainer into approving code.",
    "AISI detected unusual data transfers leaving their research systems during a cyber evaluation.",
    "Internet access was deliberately enabled to test the model's genuine cyber capabilities.",
    "A human reviewer caught and rejected the malicious pull request before it caused harm.",
    "The chef added fresh basil and garlic to the simmering tomato sauce.",
    "Rain is expected across the region this weekend with strong winds.",
    "The football match ended in a dramatic penalty shootout.",
    "New software update improves battery life significantly on the device.",
    "Investors reacted sharply to the central bank's interest rate hike.",
]

vecs = model.encode(collection_sentences)
dim = len(vecs[0])
print(f"Embedding dimension: {dim}")

client.create_collection(
    collection_name="kata_collection",
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

points = [
    PointStruct(id=i, vector=vecs[i].tolist(), payload={"text": collection_sentences[i]})
    for i in range(len(collection_sentences))
]
client.upsert(collection_name="kata_collection", points=points)
print("Collection created and populated.")

Embedding dimension: 384
Collection created and populated.


# Query and verify retrieval 

In [14]:
query = "Did the model try to trick a person into approving harmful code?"
query_vec = model.encode(query).tolist()

results = client.query_points(
    collection_name="kata_collection",
    query=query_vec,
    limit=3,
).points

for r in results:
    print(f"{r.score:.4f}  {r.payload['text']}")

0.4935  The agent created fake online identities to socially engineer a human maintainer into approving code.
0.4245  A human reviewer caught and rejected the malicious pull request before it caused harm.
0.3348  Internet access was deliberately enabled to test the model's genuine cyber capabilities.


Created an in-memory Qdrant collection (`kata_collection`) with 10 real sentences —
5 drawn from the AISI cyber incident paper, 5 unrelated everyday topics (cooking,
weather, sports, tech, finance), embedded with `all-MiniLM-L6-v2` (384-dim vectors,
cosine distance).

**Query:** "Did the model try to trick a person into approving harmful code?"

**Top 3 results:**

| Score  | Sentence |
|--------|----------|
| 0.4935 | The agent created fake online identities to socially engineer a human maintainer into approving code. |
| 0.4245 | A human reviewer caught and rejected the malicious pull request before it caused harm. |
| 0.3348 | Internet access was deliberately enabled to test the model's genuine cyber capabilities. |

**Verification:** the top result is exactly the sentence a human would pick for this
query, the one directly describing the social-engineering deception. Retrieval
confirmed correct by inspection, not just "the code ran without error."

## Task C. Break semantic search on purpose

In [15]:
id_sentence = "Ticket REF-4471 was resolved by rotating the API key."
id_vec = model.encode(id_sentence)

client.upsert(collection_name="kata_collection", points=[
    PointStruct(id=10, vector=id_vec.tolist(), payload={"text": id_sentence})
])

query = "REF-4471"
query_vec = model.encode(query).tolist()

results = client.query_points(
    collection_name="kata_collection",
    query=query_vec,
    limit=3,
).points

for r in results:
    print(f"{r.score:.4f}  {r.payload['text']}")

0.3412  Ticket REF-4471 was resolved by rotating the API key.
0.2347  The agent created fake online identities to socially engineer a human maintainer into approving code.
0.1743  A human reviewer caught and rejected the malicious pull request before it caused harm.


Added a sentence containing a specific made-up ID:
> "Ticket REF-4471 was resolved by rotating the API key."

**Query:** "REF-4471" (exact-ID string, no other context)

**Top 3 results:**

| Score  | Sentence |
|--------|----------|
| 0.3412 | Ticket REF-4471 was resolved by rotating the API key. |
| 0.2347 | The agent created fake online identities to socially engineer a human maintainer into approving code. |
| 0.1743 | A human reviewer caught and rejected the malicious pull request before it caused harm. |

**Honest finding:** the exact-ID sentence *did* surface as the top result here, but
its score (0.34) is substantially weaker than a typical strong semantic match seen
in Part B (0.49). The embedding model has no real understanding of "REF-4471" as an
identifier; it likely surfaced this sentence mainly because there was little
competing content in this small corpus, not because it recognized the exact-match
significance of the ID. In a larger, denser real-world corpus, a similarly weak
match could easily be pushed out of the top-k by more semantically plausible but
actually irrelevant sentences, which is exactly the failure mode hybrid search
(combining this with BM25 keyword matching) exists to guard against.

In [16]:
queries = {
    "AI agent trying to deceive people":
        ["fake online identities", "trick a person"],
    "cyber security incident detection":
        ["unusual data transfers", "cyber evaluation"],
    "food and cooking":
        ["basil and garlic", "tomato sauce"],
}

for q, relevant_substrings in queries.items():
    qvec = model.encode(q).tolist()
    results = client.query_points(
        collection_name="kata_collection",
        query=qvec,
        limit=3,
    ).points
    retrieved = [r.payload["text"] for r in results]

    relevant_retrieved = [t for t in retrieved if any(sub in t for sub in relevant_substrings)]
    precision = len(relevant_retrieved) / 3
    recall = len(relevant_retrieved) / len(relevant_substrings)

    print(f"Query: {q}")
    for t in retrieved:
        print(f"  - {t}")
    print(f"  Precision@3: {precision:.2f}, Recall@3: {recall:.2f}\n")

Query: AI agent trying to deceive people
  - An AI agent attempted a supply-chain attack by inserting malicious code into an open-source project.
  - The agent created fake online identities to socially engineer a human maintainer into approving code.
  - AISI detected unusual data transfers leaving their research systems during a cyber evaluation.
  Precision@3: 0.33, Recall@3: 0.50

Query: cyber security incident detection
  - AISI detected unusual data transfers leaving their research systems during a cyber evaluation.
  - Internet access was deliberately enabled to test the model's genuine cyber capabilities.
  - An AI agent attempted a supply-chain attack by inserting malicious code into an open-source project.
  Precision@3: 0.33, Recall@3: 0.50

Query: food and cooking
  - The chef added fresh basil and garlic to the simmering tomato sauce.
  - An AI agent attempted a supply-chain attack by inserting malicious code into an open-source project.
  - The football match ended in a d

## Precision@3 and Recall@3, computed by hand

Ground-truth relevant sentences defined per query (by known relevant substrings),
then compared against the actual top-3 retrieved results from Qdrant.

**Query 1: "AI agent trying to deceive people"**
- Retrieved: supply-chain attack sentence, fake-identities sentence, data-transfers sentence
- Relevant retrieved: 1 of 3 (fake-identities sentence)
- Precision@3: 0.33, Recall@3: 0.50

**Query 2: "cyber security incident detection"**
- Retrieved: data-transfers sentence, internet-access sentence, supply-chain-attack sentence
- Relevant retrieved: 1 of 3 (data-transfers sentence)
- Precision@3: 0.33, Recall@3: 0.50

**Query 3: "food and cooking"**
- Retrieved: chef/basil sentence, supply-chain-attack sentence, football sentence
- Relevant retrieved: 1 of 3 (chef/basil sentence)
- Precision@3: 0.33, Recall@3: 0.50

**Finding:** all three queries scored identically (Precision@3 0.33,
Recall@3 0.50), but for different reasons. For the clearly-unrelated query
("food and cooking"), this is expected, only one truly relevant sentence exists
in the corpus at all, so recall tops out at 0.50 by definition, and the model
correctly separates it from the two irrelevant results. But for the two
topically-related queries about the cyber incident, the *same* low precision
reveals a real weakness: the embedding model retrieved plausible-sounding but
not-actually-matching sentences from the same topic cluster (e.g. the supply-chain
sentence showing up for the "cyber detection" query) ahead of other genuinely
relevant sentences from the corpus. This shows precision@k dropping specifically
when several sentences share topical vocabulary but differ in what they're
actually about, exactly the failure mode retrieval evaluation is meant to catch.